In [ ]:
from pathlib import Path
from io import BytesIO
import json, csv, hashlib, math

PACKAGE_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
                     if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError("Open this notebook from the package root or notebooks directory.")

SAVE_PDF = False

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import t
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from IPython.display import display, Image
plt.rcdefaults()

def load_npz(relative_path):
    with np.load(PACKAGE_ROOT / relative_path, allow_pickle=False) as archive:
        return {key: archive[key].copy() for key in archive.files}

def add_panel_labels(figure):
    panels = [axis for axis in figure.axes if axis.get_label() != "<colorbar>"]
    assert len(panels) == 6, "Unexpected panel count"
    for index, axis in enumerate(panels):
        axis.text(-0.16, 1.04, f"({chr(97 + index)})",
                  transform=axis.transAxes, fontsize=10, fontweight="bold",
                  ha="left", va="bottom", clip_on=False, gid="panel_label")
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    records = []
    for axis in panels:
        label = next(text for text in axis.texts if text.get_gid() == "panel_label")
        box = label.get_window_extent(renderer)
        for title in (axis.title, axis._left_title, axis._right_title):
            if title.get_text():
                assert not box.overlaps(title.get_window_extent(renderer)), "Panel label overlaps title"
        records.append({"label": label.get_text(), "axes_position": list(axis.get_position().bounds),
                        "label_bbox_pixels": list(box.bounds)})

def export_figure(figure, figure_id, tight=False):
    add_panel_labels(figure)
    options = {"bbox_inches": "tight"} if tight else {}
    preview = BytesIO()
    figure.savefig(preview, format="png", dpi=180, **options)
    display(Image(data=preview.getvalue()))
    if SAVE_PDF:
        output = PACKAGE_ROOT / "outputs"
        output.mkdir(exist_ok=True)
        figure.savefig(output / f"{figure_id}.pdf",
                       metadata={"CreationDate": None, "ModDate": None}, **options)
        print(f"Saved outputs/{figure_id}.pdf")


In [ ]:
from typing import Any
N_SEEDS = 20
SEEDS = np.arange(42, 62, dtype=np.int64)
T_CRIT_95_DF19 = 2.093024054408263
STEPS = np.arange(1001, dtype=np.int64)
SELECTED_FREQUENCIES = (1, 2, 3, 7, 15, 18)
CONDITIONS = (
    ("0p10", 0.10, "#4C78A8", "-"),
    ("0p25", 0.25, "#D6A51D", "--"),
    ("0p40", 0.40, "#E66101", "-."),
    ("0p50", 0.50, "#7A8B3A", ":"),
)
BANDS = (
    ("Low 1-5", 1, 5, "#009E73", "-"),
    ("Middle 6-15", 6, 15, "#0072B2", "--"),
    ("High 16-20", 16, 20, "#003B5C", "-."),
)
def require(condition: bool, message: str) -> None:
    if not condition:
        raise RuntimeError(message)

def mean_ci(values: np.ndarray, axis: int = 0) -> tuple[np.ndarray, np.ndarray]:
    values = np.asarray(values, dtype=np.float64)
    require(values.shape[axis] == N_SEEDS, "95% t interval requires 20 seeds")
    mean = np.mean(values, axis=axis)
    half = T_CRIT_95_DF19 * np.std(values, axis=axis, ddof=1) / math.sqrt(N_SEEDS)
    return mean, half

def setup_style() -> None:
    plt.rcParams.update(
        {
            "font.family": "serif",
            "font.serif": ["DejaVu Serif"],
            "font.size": 8.5,
            "axes.titlesize": 8.5,
            "axes.labelsize": 8.5,
            "xtick.labelsize": 7.5,
            "ytick.labelsize": 7.5,
            "legend.fontsize": 8.0,
            "axes.linewidth": 0.65,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "savefig.facecolor": "white",
            "figure.facecolor": "white",
            "axes.facecolor": "white",
        }
    )

def plot_fig3_style(data: dict[str, dict[str, Any]], output_dir: Path) -> list[str]:
    setup_style()
    figure, axes = plt.subplots(2, 3, figsize=(7.2, 5.15), sharex=True, sharey=True)
    for axis, frequency in zip(axes.flat, SELECTED_FREQUENCIES):
        for tag, learning_rate, color, linestyle in CONDITIONS:
            mean = np.mean(data[tag]["amplitude"][:, :, frequency], axis=0)
            line, = axis.plot(
                STEPS,
                mean,
                label=rf"$\eta={learning_rate:.2f}$",
                color=color,
                linestyle=linestyle,
                linewidth=1.05,
            )
            require(np.array_equal(line.get_ydata(), mean), f"Fig. 3 binding mismatch for {tag}/k={frequency}")
        axis.set_title(r"$k=" + str(frequency) + "$", loc="left", pad=4)
        axis.set_ylim(-0.001, 0.023)
        axis.set_xticks([0, 200, 400, 600, 800, 1000])
        axis.ticklabel_format(axis="y", style="plain", useOffset=False)
        decorate(axis)
    for axis in axes[1]:
        axis.set_xlabel("Training iteration", labelpad=3)
    for axis in axes[:, 0]:
        axis.set_ylabel("Mean Fourier amplitude", labelpad=3)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    figure.legend(
        handles,
        labels,
        ncol=4,
        loc="upper center",
        bbox_to_anchor=(0.52, 0.995),
        frameon=False,
        fontsize=8.5,
        columnspacing=2.2,
        handlelength=2.8,
        handletextpad=0.6,
    )
    figure.subplots_adjust(left=0.09, right=0.97, bottom=0.10, top=0.87, hspace=0.35, wspace=0.20)
    return save_figure(figure, output_dir, "Fig3_style_noiseless_learning_rate_comparison_GD20")

def plot_fig4_style(data: dict[str, dict[str, Any]], output_dir: Path) -> list[str]:
    setup_style()
    figure, axes = plt.subplots(1, 4, figsize=(8.4, 3.15), sharex=True, sharey=True)
    for axis, (tag, learning_rate, _, _) in zip(axes, CONDITIONS):
        for label, _, _, color, linestyle in BANDS:
            values = data[tag]["band_ratios"][label]
            mean, half = mean_ci(values)
            line, = axis.plot(STEPS, mean, color=color, linestyle=linestyle, linewidth=1.0, label=label)
            require(np.array_equal(line.get_ydata(), mean), f"Fig. 4 binding mismatch for {tag}/{label}")
            axis.fill_between(
                STEPS,
                np.maximum(mean - half, 1e-4),
                mean + half,
                color=color,
                alpha=0.13,
                linewidth=0,
            )
        axis.axhline(1.0, color="0.35", linewidth=0.65, linestyle=(0, (3, 2)), zorder=1)
        axis.set_title("Noiseless\n" + rf"$\eta={learning_rate:.2f}$", fontsize=8.5, pad=7)
        axis.set_yscale("log")
        axis.set_ylim(1e-4, 1.3)
        axis.set_xticks([0, 500, 1000])
        axis.grid(alpha=0.12, which="both")
        decorate(axis)
        axis.set_xlabel("Training iteration", labelpad=3)
    axes[0].set_ylabel(r"Mean relative band loss $R_B(n)$", labelpad=3)
    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.53, 0.995),
        ncol=3,
        frameon=False,
        fontsize=8.5,
        handlelength=2.8,
        columnspacing=1.7,
    )
    figure.subplots_adjust(left=0.075, right=0.985, bottom=0.19, top=0.72, wspace=0.19)
    return save_figure(figure, output_dir, "Fig4_style_noiseless_learning_rate_comparison_GD20")
def save_figure(figure,output_dir,stem):
    export_figure(figure,"FigS10")
    plt.close(figure)
    return []

def decorate(axis: plt.Axes) -> None:
    axis.set_xlim(0, 1000)
    axis.tick_params(direction="out", length=2.5, width=0.65, pad=2)

In [ ]:
data={}
for tag,step_size,_,_ in CONDITIONS:
    raw=load_npz(f"data/FigS10_FigS11/{tag}.npz")
    assert raw["amplitude"].shape==(20,1001,40)
    bands={}
    for label,lo,hi,_,_ in BANDS:
        losses=raw["mode_loss"][:,:,lo:hi+1].sum(-1)
        bands[label]=losses/losses[:,:1]
    data[tag]={"amplitude":raw["amplitude"],"band_ratios":bands}

In [ ]:
plot_fig3_style(data,None)